In [1]:
import pandas as pd
import json 
import altair as alt
import glob

In [2]:
summary = pd.read_csv('results/exact/summary.csv')
summary

,solver,sample_stops,seed,n_stops,status,objective,best_bound,gap,operator_cost,user_cost,buses,city_time_min,length_km,avg_user_time_min,solve_time_s,route
0,HIGHS,NaN,0,2,OPTIMAL,8453.628972,8453.628972,0.0,1397.205772,15510.052172,5,11.853165,3.555950,46.465105,9.257480,SM-214 -> SM-328
1,HIGHS,NaN,0,3,OPTIMAL,8376.150529,8376.150529,0.0,1398.071943,15354.229115,5,17.102683,5.130805,45.998290,86.947693,SM-214 -> SM-328 -> SB-1390
2,HIGHS,NaN,0,4,OPTIMAL,8318.470234,8318.470234,0.0,1678.751776,14958.188692,6,26.980463,8.094139,44.811830,1356.299311,SM-214 -> SM-328 -> SB-483 -> SF-BN
3,HIGHS,NaN,0,5,OPTIMAL,8325.245714,8325.245714,0.0,1679.385406,14971.106022,6,30.820642,9.246193,44.850527,1923.576196,SM-214 -> SM-328 -> SB-1390 -> SF-BN -> ST-MRCR


In [7]:
# create a simple regression model with y = solve_time_s and x = n_stops
from sklearn.linear_model import LinearRegression
X = summary[['n_stops']]
y = summary['solve_time_s']
model = LinearRegression()
model.fit(X, y)
# predict the solve_time_s for n_stops = 6,7,8,9, 10 and plot the predicted values
n_stops_values = [6, 7, 8, 9, 10]
predicted_solve_times = model.predict([[n] for n in n_stops_values])
# plot
alt.Chart(pd.DataFrame({'n_stops': n_stops_values, 'predicted_solve_time_s': predicted_solve_times})).mark_line(color='red').encode(
    x='n_stops:Q',
    y='predicted_solve_time_s',
    tooltip=['n_stops', 'predicted_solve_time_s']
).properties(
    width=600,
    height=400
) + alt.Chart(summary).mark_line().encode(
    x='n_stops:Q',
    y='solve_time_s',
    tooltip=['n_stops', 'solve_time_s']
).properties(
    width=600,
    height=400
)

/Users/bmantepe/Library/Python/3.12/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


alt.LayerChart(...)

In [4]:
# line plot x = n_stops, y = solve_time_s 
overall = alt.Chart(summary).mark_line().encode(
    x=alt.X('n_stops:O', title='Number of Stops'),
    y=alt.Y('objective', title='Objective Value',scale=alt.Scale(domain=[summary['objective'].min() - 100, summary['objective'].max() + 100])),
    tooltip=['n_stops', 'objective']
).properties(
    width=600,
    height=400,title='Overall Objective Value')

# line plot x = n_stops, y = solve_time_s 
operator = alt.Chart(summary).mark_line().encode(
    x=alt.X('n_stops:O', title='Number of Stops'),
    y=alt.Y('operator_cost', title='Objective Value',scale=alt.Scale(domain=[summary['operator_cost'].min() - 100, summary['operator_cost'].max() + 100])),
    tooltip=['n_stops', 'operator_cost']
).properties(
    width=600,
    height=400,title='Operator Cost')

user = alt.Chart(summary).mark_line().encode(
    x=alt.X('n_stops:O', title='Number of Stops'),
    y=alt.Y('user_cost', title='Objective Value',scale=alt.Scale(domain=[summary['user_cost'].min() - 100, summary['user_cost'].max() + 100])),
    tooltip=['n_stops', 'user_cost']
).properties(
    width=600,
    height=400,title='User Cost')

overall | operator | user

alt.HConcatChart(...)